# Multi-Agent Systems with Google Agent Development Kit (ADK)

**Challenge 4**: Hierarchical Orchestration, Lifecycle Callbacks & Loop Workflows**

---

## 🚀 Key Features

* 🤖 **Hierarchical Multi-Agent Architecture:** Root coordinator agent managing specialized sub-agents via delegation and `AgentTool`.
* 🌦️ **Weather Specialist:** Real-time forecast retrieval and coordinate geocoding via the National Weather Service (NWS) API.
* 🔍 **Built-in Google Search Grounding:** Real-time web retrieval using ADK's native `google_search` tool.
* 🎬 **Iterative Film Production Assembly Line:** Autonomous `LoopAgent` pipeline featuring Search, Critique, and Refine agents with state tracking.
* 🛡️ **Lifecycle Callbacks & Moderation:** `before_model` and `after_model` interceptors for location constraints, safety filtering, and request/response telemetry.
* 🧪 **End-to-End Test Suite:** Streamed execution and event logging across multi-turn sessions using `InMemoryRunner` and `AdkApp`.

## Step 1: Install Dependencies

In [1]:
# !pip install google-adk google-genai requests python-dotenv nest-asyncio -q
%pip install "google-adk[extensions]" litellm google-genai requests python-dotenv nest-asyncio -q


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 2: Import Libraries

In [2]:
import os
import json
import requests
import asyncio
import random
import uuid
from typing import Dict, Any, Optional

# Enable nested event loops for Jupyter
import nest_asyncio
nest_asyncio.apply()

# Google ADK imports
from google.adk.agents import Agent, LoopAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, ToolContext, google_search
from google.genai.types import Content, Part

from IPython.display import Markdown, display

import vertexai
from vertexai.preview import reasoning_engines

from dotenv import load_dotenv
load_dotenv()

# Set model
MODEL_NAME = "gemini-2.5-flash"


# State management tool matching the slides
def append_to_state(
    tool_context: ToolContext, field: str, response: str
) -> dict:
    """Appends responses into the shared workflow state dictionary."""
    existing_state = tool_context.state.get(field, [])
    tool_context.state[field] = existing_state + [response]
    return {"status": "success"}

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## Step 3: Configuration

Set your API keys here (optional - notebook works without them for common cities)

In [3]:
# API Keys
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY", "")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

# Project configuration
PROJECT_ID = "qwiklabs-gcp-02-138827e82db5"
LOCATION = "us-central1"

# Set environment variables for Vertex AI / Google GenAI SDK
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

# Initialize Vertex AI globally
vertexai.init(project=PROJECT_ID, location=LOCATION)

print("✅ Configuration loaded and Vertex AI initialized")

✅ Configuration loaded and Vertex AI initialized


## Step 2: Define Sub-Agents (Search, Critique, Refine)

In [4]:
# (b) Search Agent: Finds facts, inspiration, and plot premises
search_agent = Agent(
    name="search_agent",
    model=MODEL_NAME,
    description="Searches for real-world background data, historical contexts, and narrative inspirations.",
    instruction="""You are a movie researcher on the film production assembly line.
Use the google_search tool to gather factual details, cinematic references, or plot ideas based on the user's concept.
Summarize your findings clearly for the creative team.""",
    tools=[google_search],
)

# (c) Critique Agent: Evaluates the draft and suggests improvements
critique_agent = Agent(
    name="critique_agent",
    model=MODEL_NAME,
    description="Reviews current film concept drafts and provides constructive critiques and suggestions.",
    instruction="""You are an expert film critic and story editor on the assembly line.
Evaluate the current movie concept draft and search findings.
Identify pacing issues, plot holes, cliché dialogue, or character development gaps.
Provide 2-3 specific, actionable suggestions for improvement.""",
    tools=[append_to_state],
)

# (d) Refine Agent: Rewrites and polishes the script/plot concept
refine_agent = Agent(
    name="refine_agent",
    model=MODEL_NAME,
    description="Rewrites and improves the movie plot concept incorporating critique suggestions.",
    instruction="""You are the lead screenwriter on the assembly line.
Take the existing plot draft and the feedback from the critique_agent.
Rewrite and elevate the film logline, synopsis, and scene beat sheet to address all suggested improvements.
Deliver a polished and compelling final concept.""",
    tools=[append_to_state],
)

## Step 3: Build Loop Agent & Greeter Agent

In [5]:
# Loop Agent: Iterative writers room refining the script/concept
writers_room = LoopAgent(
    name="writers_room",
    description="Iteratively searches, critiques, and refines the film concept across production cycles.",
    sub_agents=[
        search_agent,
        critique_agent,
        refine_agent,
    ],
    max_iterations=2,
)

# Greeter Root Agent: Entry point for user interactions
GREETER_INSTRUCTIONS = """You are the executive producer and greeter for the Film Production Assembly Line.
When a user provides a movie prompt, pitch, or concept:
1. Welcome them to the studio assembly line.
2. Delegate the concept to the writers_room loop team to develop, critique, and polish the story.
3. Deliver the final approved production-ready treatment clearly to the user."""

root_agent = Agent(
    name="greeter",
    model=MODEL_NAME,
    description="Craft a movie plot.",
    instruction=GREETER_INSTRUCTIONS,
    tools=[append_to_state],
    sub_agents=[writers_room],
)

# Wrap root agent into AdkApp & Runner
app = reasoning_engines.AdkApp(agent=root_agent)
runner = InMemoryRunner(agent=root_agent, app_name="Film Production Studio")

print(
    "✅ Film production assembly line agents and LoopAgent created successfully"
)

/var/folders/79/kkzhxd153fs9svz_j_wj0xfr0000gn/T/ipykernel_2515/1485925198.py:2: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  writers_room = LoopAgent(
App "Film Production Studio" can transfer between agents but has no context_cache_config. Every transfer swaps the system instruction and the tool set, so the request prefix changes and the whole prompt is re-sent uncached after each transfer. Set context_cache_config on the app to give each agent its own cache.


✅ Film production assembly line agents and LoopAgent created successfully


## Step 4: Create Session

In [6]:
user_id = "producer-user-1"
session = app.create_session(user_id=user_id)
session_id = session.get("id") if isinstance(session, dict) else session.id

print(f"🎬 Production Session ID: {session_id}")

/Users/ridwan/.local/share/virtualenvs/docscan-wrm2pkcA/lib/python3.13/site-packages/vertexai/preview/reasoning_engines/templates/adk.py:975: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/Users/ridwan/.local/share/virtualenvs/docscan-wrm2pkcA/lib/python3.13/site-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
App "default-app-name" can transfer between agents but has no context_cache_config. Every transfer swaps the system instruction and the tool set, so the request prefix changes and the whole prompt is re-sent uncached after e

🎬 Production Session ID: 81befa35-d061-432c-b7e1-5ac593ed2203


## Step 5: Test Film Production Multi-Agent Assembly Line

In [7]:
def test_film_assembly_line(prompt: str):
    print(f"\n{'='*80}")
    print(f"🎬 NEW FILM PITCH: {prompt}")
    print(f"{'='*80}")

    last_event = None
    try:
        for event in app.stream_query(
            user_id=user_id,
            session_id=session_id,
            message=prompt,
        ):
            last_event = event

            # Track agent handoffs and loop steps
            if isinstance(event, dict):
                author = event.get("author") or event.get("agent_name", "")
                actions = event.get("actions", {})
                if author:
                    print(f"  🔄 [Active Station: {author}]")
                if actions and actions != {
                    "state_delta": {},
                    "artifact_delta": {},
                    "requested_auth_configs": {},
                    "requested_tool_confirmations": {},
                }:
                    print(f"     ⚙️ Action: {actions}")

        # Render final refined film package
        if (
            last_event
            and isinstance(last_event, dict)
            and "content" in last_event
            and last_event["content"]
            and "parts" in last_event["content"]
            and len(last_event["content"]["parts"]) > 0
        ):
            print("\n🎞️ FINAL PRODUCTION TREATMENT:")
            display(Markdown(last_event["content"]["parts"][0]["text"]))
        else:
            print("\n⚠️ No final output received from assembly line.")

    except Exception as e:
        print(f"\n❌ Pipeline execution failed: {str(e)}")


# Run Test Pitch
test_film_assembly_line(
    "Pitch a neo-noir sci-fi mystery set in underwater Tokyo in the year 2088."
)


🎬 NEW FILM PITCH: Pitch a neo-noir sci-fi mystery set in underwater Tokyo in the year 2088.


/Users/ridwan/.local/share/virtualenvs/docscan-wrm2pkcA/lib/python3.13/site-packages/google/adk/tools/transfer_to_agent_tool.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  function_decl = super()._get_declaration()
Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


  🔄 [Active Station: greeter]
  🔄 [Active Station: greeter]
     ⚙️ Action: {'state_delta': {}, 'artifact_delta': {}, 'transfer_to_agent': 'writers_room', 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}
  🔄 [Active Station: search_agent]
  🔄 [Active Station: critique_agent]
  🔄 [Active Station: refine_agent]
  🔄 [Active Station: refine_agent]
     ⚙️ Action: {'state_delta': {'logline': ['In the submerged, neon-drenched districts of Tokyo, 2088, a disgraced deep-sea structural engineer, haunted by a catastrophic past failure, must navigate a labyrinthine conspiracy involving corporate greed, black market bio-engineering, and explosive secrets from a pre-flood past to clear his name and prevent a new, city-wide catastrophe that threatens to drown all hope.'], 'synopsis': ['The story begins when Hiroshi Tanaka, a high-ranking executive spearheading the lucrative "Re-Stabilization Project" – an ambitious, controversial initiative to build new, exclusive habitat domes deep

The refined concept for "Aqua-Noir" is now complete, integrating all feedback to create a robust and compelling neo-noir sci-fi mystery.

### Project: Aqua-Noir

**Logline:** In the submerged, neon-drenched districts of Tokyo, 2088, a disgraced deep-sea structural engineer, haunted by a catastrophic past failure, must navigate a labyrinthine conspiracy involving corporate greed, black market bio-engineering, and explosive secrets from a pre-flood past to clear his name and prevent a new, city-wide catastrophe that threatens to drown all hope.

**Synopsis:** The story begins when Hiroshi Tanaka, a high-ranking executive spearheading the lucrative "Re-Stabilization Project" – an ambitious, controversial initiative to build new, exclusive habitat domes deeper in the ocean – is found dead. The official report blames a structural malfunction, a tragic accident of deep-sea living. But Kaito, a disgraced former deep-sea structural engineer now working as a private investigator for the forgotten, senses something far more sinister. Kaito carries the heavy burden of a past deep-sea disaster: a catastrophic dome collapse he was unfairly blamed for, leading to countless lives lost and his banishment from corporate projects. This history makes him uniquely attuned to the subtle signs of engineered failure and corporate negligence beneath the waves. The pervasive reach of the corporate powers is immediately felt; Kaito's own low-stakes recovery jobs in the Drip Zones are often subtly interfered with by corporate drones, serving as a constant reminder of who truly controls the currents, and an anonymous warning, perhaps a corrupted data packet left in his submersible, advises him to "let sleeping currents lie" in the wake of Tanaka's death.

Kaito, operating from a jury-rigged submersible, is reluctantly approached by Dr. Anya Tanaka, the executive’s estranged daughter. Anya isn't merely seeking justice; she’s a brilliant but disillusioned bio-engineer who worked alongside her father on the very project that killed him. Her initial demeanor is visibly strained, hinting at a deep internal conflict. She possesses fragments of his encrypted research data, which hint that her father was murdered because he discovered a fatal flaw in the "Re-Stabilization Project": the new domes are being constructed directly over a volatile, pre-flood geological fault line. Their "structural integrity," she reveals, relies on illegally modified deep-sea organisms, some of which she herself helped develop the foundational biotech for, before realizing their malevolent application by the corporation. Anya admits her father was close to exposing how corporations are deliberately destabilizing existing lower-tier structures (the "Drip Zones") – fabricating "accidents" – to accelerate the need for these new, profitable domes, and to suppress any opposition using these modified organisms to reinforce or destroy at will. Anya’s own motives are complex; she seeks not only to avenge her father but also to protect her own compromised research and legacy, making her a crucial yet morally ambiguous ally, whose past ethical compromises make Kaito question her at every turn.

As Kaito delves deeper, navigating the treacherous currents of both the ocean and corporate espionage, he uncovers:

*   **Corporate Collusion & Environmental Exploitation:** Evidence that the "Re-Stabilization Project" is a façade for a ruthless power grab. Corporations are intentionally engineering environmental threats and structural failures in the "Drip Zones" to justify expensive new construction, displace lower-tier citizens, and exploit resources from vulnerable deep-sea ecosystems under the guise of "re-stabilization."
*   **Biotech Black Market & Unethical Experiments:** The illegal cultivation and rapid modification of bioluminescent organisms and extreme-pressure flora/fauna, not only to reinforce the unstable new domes but also for use in covert operations – potentially as biological weapons or tools for sabotage. Tanaka's research fragments indicate these modifications have unforeseen and deadly side effects on both human physiology and the fragile deep-sea environment, a truth his father was trying to expose, perhaps even having been involved in early, unethical human trials himself. Anya's complicity in the initial stages of this research adds a layer of personal guilt and a dangerous secret to their alliance.
*   **A Pre-Flood Catastrophe & Suppressed Technology:** Clues link the "fault line" beneath the new domes to not just a geological anomaly, but a forgotten, active ancient energy source or technological nexus, hinting at a historical deep-sea catastrophe that mirrors Kaito's own past failure. This revelation exposes a cyclical pattern of corporate hubris and deliberate suppression of dangerous knowledge spanning generations. The location of the new domes is no accident; it’s a direct, calculated exploitation of this powerful, unstable pre-flood secret.

The mastermind behind this conspiracy is **Chairman Ryota Kuroda**, the stoic and unyielding CEO of Oceanic Futures Corp., the parent company of the "Re-Stabilization Project." Kuroda isn't merely greedy; he harbors a messianic belief that humanity is a virus that has destroyed the surface world and that the only way to ensure its survival, and its evolution, is to force it to adapt to the deep ocean, under his absolute control. His family suffered unimaginable losses in the Great Floods, leaving him with an unshakeable conviction that only through extreme control and bio-engineered supremacy can humanity endure. He views the pre-flood past not as a warning, but as a blueprint for a more "efficient," albeit ruthless, survival strategy, believing that the ancient energy source and modified organisms are tools for humanity's necessary, brutal rebirth. He sees Hiroshi Tanaka's ethical qualms as weakness, a threat to his vision.

Kaito's investigation forces him to confront not only Kuroda's formidable power and twisted ideology but also his own past demons, as the lines blur between saving the city and seeking personal redemption. Anya, grappling with her own ethical compromises, becomes a crucial, albeit complicated, ally, forcing Kaito to question who he can truly trust in this submerged world, and ultimately confronting her role in enabling the very dangers they now face.

**Scene Beat Sheet: Aqua-Noir**

1.  **Opening: The Drip Zone's Grim Reality & Early Warning.** Establish the claustrophobic atmosphere of the "Drip Zones." Kaito, haggard, navigates a treacherous sector in his rundown submersible, on a low-stakes recovery. His sub's systems are momentarily interfered with by a corporate drone, hinting at pervasive surveillance. News reports (sub-aquatic broadcasts) hint at the "Re-Stabilization Project" and the "tragic accident" of executive Hiroshi Tanaka. Kaito dismisses it, jaded. A corrupted data packet pings his console: "Let sleeping currents lie. Some truths are better left submerged." This anonymous threat directly references his past. A haunting flashback: Kaito, younger, frantically trying to save lives during a catastrophic dome collapse – a disaster he was blamed for, but knew was an engineered flaw.
2.  **The Compromised Client:** Dr. Anya Tanaka, visibly strained and conflicted, seeks Kaito. She dismisses the official report, claiming her father was murdered. She offers not just credits, but encrypted fragments of her father's research – data that could indirectly clear Kaito’s name by exposing the corporate entity responsible for *both* disasters. She mentions "structural anomalies" and "unnatural bioluminescence" at the new dome sites. Her own past involvement in foundational bio-tech research that was twisted by the corporation is hinted at through her visible guilt and guarded responses about her work. Kaito senses her moral compromise, deepening his suspicion of her motives, but the potential for redemption, both for himself and for the city, pulls him in.
3.  **Initial Infiltration & Overt Corporate Hostility:** Kaito reluctantly takes the case. His first attempts to investigate the accident site are met with *immediate* and aggressive corporate security submersibles, forcing a dangerous chase through deep-sea canyons. This overt hostility confirms the anonymous warning and solidifies his suspicion of a cover-up. He finds subtle discrepancies at the site – unusual pressure readings, signs of tampering disguised as "structural failure" – patterns eerily similar to his past catastrophe.
4.  **The Biotech Link & Black Market Run-in:** Kaito’s investigation leads him to the "Drip Zones" black market, specifically an illegal bio-lab. He’s looking for the source of modified deep-sea organisms related to Anya's clue. He discovers these organisms are being secretly developed for "structural reinforcement" in the deeper, unstable zones of the new domes, but also for other, more sinister purposes. He narrowly escapes an ambush by corporate enforcers, who are also exploiting these illicit operations, revealing the corporate ties to the underworld.
5.  **Anya's Full Confession & The Fault Line's Secret:** Kaito confronts Anya with his findings and her evasiveness. Under pressure, she fully confesses her past involvement in developing the core bio-tech that the corporation then perverted. She reveals her father was horrified by the unethical nature of the project and that the new domes are being built directly over a highly volatile, pre-flood geological fault line. She reveals her father was using her research to stabilize it, blurring his own moral lines, and was close to exposing the corporations' deliberate sabotage of the Drip Zones to create "demand" for these hazardous new habitats. Her confession, though painful, earns a sliver of Kaito's trust.
6.  **The Pre-Flood Catastrophe & Kuroda's Ideology:** Kaito, using his deep-sea knowledge, accesses restricted archives of the sunken city. He finds records of the very deep-sea disaster that ruined him, realizing his past failure was likely a precursor or test-run for the current "Re-Stabilization Project," implying corporate knowledge and deliberate suppression. He discovers that the "fault line" is actually a dormant, but easily reactivated, ancient energy conduit or a "nexus point" for the bio-engineered organisms. Further investigation reveals the personal history of **Chairman Ryota Kuroda**, CEO of Oceanic Futures Corp. Kuroda's family suffered greatly in the Great Floods, fueling his messianic belief that only through absolute control of this ancient power and bio-engineered adaptation can humanity truly survive and evolve. He views the pre-flood past not as a warning, but a blueprint for a necessary, brutal rebirth.
7.  **The Mastermind's Grand Scheme:** Kaito and Anya now understand Kuroda's full, terrifying vision. He orchestrated Tanaka's murder to prevent him from exposing the environmental and societal catastrophe his project would cause. His goal isn't just new domes; it's total control over the city's power and resources, using the pre-flood energy source and bio-engineered organisms as weapons and tools of coercion, reshaping humanity itself. They've been deliberately creating instability in the Drip Zones to manufacture "demand" and weed out the "weak."
8.  **Climax - Race Against the Collapse:** Kaito and Anya must race against time to expose the conspiracy before the "Re-Stabilization Project" goes fully online, triggering a city-wide environmental collapse and solidifying Kuroda's regime. This involves infiltrating a deep-sea corporate facility where the bio-engineered organisms are being deployed to activate the ancient energy source. Kaito uses his unique deep-sea skills and knowledge of structural weaknesses to navigate the treacherous environment. Anya, grappling with her past ethical compromises, uses her bio-engineering expertise to counter the modified organisms and expose Kuroda's full plan to the city, fighting for a chance at redemption.
9.  **Resolution:** The conspiracy is exposed, but not without significant personal and environmental cost. The city is saved from immediate collapse, but the fragile ecosystem is still threatened, and Kuroda's twisted ideology leaves a lasting scar. Kaito achieves a hard-won redemption, having prevented a disaster far worse than his past, his faith in some form of justice rekindled. Anya, her father's legacy cleared and her own complicity exposed, faces a long road to atonement, perhaps dedicating herself to ethical bio-engineering for the city's true survival, forever changed by her ordeal.